# 02 - Qualité nutritionnelle de l'offre alimentaire en France

**Objectif 1** : dresser le portrait nutritionnel des produits vendus en France à partir d'Open Food Facts.

Questions traitées :
1. Comment se répartissent le Nutri-Score et le groupe NOVA ?
2. Un produit ultra-transformé est-il forcément mal noté par le Nutri-Score ?
3. Quels additifs sont les plus fréquents, et sont-ils liés à la transformation ?
4. Quels rayons concentrent sucre, sel et graisses saturées ?

Chaque calcul porte sur les produits qui disposent de l'information nécessaire : l'effectif est indiqué sous chaque graphique. Les titres des graphiques sont calculés à partir des données.

Prérequis : `uv run python -m nutrilys.data.clean`.

In [1]:
from nutrilys.analysis import nutrition
from nutrilys.analysis.stats import cramers_v, describe, spearman
from nutrilys.db import products_connection
from nutrilys.viz import charts
from nutrilys.viz.fmt import fmt_int, fmt_num, fmt_pct
from nutrilys.viz.theme import NOVA_COLORS, NOVA_LABELS, NUTRISCORE_COLORS, save_figure, use_theme

use_theme()
con = products_connection()
findings = []  # constats repris dans la synthèse finale

## 1. Répartition du Nutri-Score

Le Nutri-Score classe la qualité nutritionnelle de A (meilleure) à E. Il s'agit ici de l'algorithme révisé en 2023, tel que calculé par Open Food Facts.

In [2]:
ns = nutrition.nutriscore_distribution(con)
share = ns.set_index("nutriscore_grade")["share"]
n_ns = ns["products"].sum()
share_de, share_ab = share[["d", "e"]].sum(), share[["a", "b"]].sum()

fig = charts.category_bars(
    ns,
    "nutriscore_grade",
    "share",
    NUTRISCORE_COLORS,
    counts="products",
    title=f"{fmt_pct(share_de)} des produits notés sont classés D ou E, {fmt_pct(share_ab)} A ou B",
    subtitle=f"Répartition du Nutri-Score, {fmt_int(n_ns)} produits vendus en France",
)
fig.update_xaxes(title_text="Nutri-Score")
save_figure(fig, "nutriscore_distribution")
findings.append(f"Nutri-Score : {fmt_pct(share_de)} de D ou E, {fmt_pct(share_ab)} de A ou B.")
fig.show()

## 2. Répartition des groupes NOVA

La classification NOVA ne mesure pas la composition nutritionnelle mais le **degré de transformation** : de 1 (aliments bruts ou peu transformés) à 4 (produits ultra-transformés, contenant des substances rarement utilisées en cuisine : arômes, émulsifiants, édulcorants...).

In [3]:
nova = nutrition.nova_distribution(con)
share_nova = nova.set_index("nova_group")["share"]
n_nova = nova["products"].sum()

fig = charts.category_bars(
    nova,
    "nova_group",
    "share",
    NOVA_COLORS,
    labels=NOVA_LABELS,
    counts="products",
    title=f"{fmt_pct(share_nova[4])} des produits renseignés sont ultra-transformés (NOVA 4)",
    subtitle=f"Répartition des groupes NOVA, {fmt_int(n_nova)} produits avec liste d'ingrédients",
)
save_figure(fig, "nova_distribution")
findings.append(f"NOVA : {fmt_pct(share_nova[4])} de produits ultra-transformés (groupe 4).")
fig.show()

**À garder en tête** : NOVA n'est calculable que si la liste d'ingrédients est renseignée, soit environ un produit sur quatre. Cette sous-population n'est pas forcément représentative de tout le marché (voir les limites en fin de notebook).

## 3. Nutri-Score et NOVA : deux mesures différentes

Le Nutri-Score évalue la composition (sucre, sel, graisses, fibres...), NOVA la transformation. Un produit peut donc être bien noté par l'un et mal par l'autre. Ce croisement est au cœur de l'objectif « controverses ».

In [4]:
cross = nutrition.nutriscore_by_nova(con)
table = nutrition.contingency(cross, "nova_group", "nutriscore_grade", "products")
assoc = cramers_v(table)

nova4 = cross[cross["nova_group"] == 4].set_index("nutriscore_grade")["share"]
nova4_ab = nova4.reindex(["a", "b"]).fillna(0).sum()
n_nova4_ab = cross.query("nova_group == 4 and nutriscore_grade in ['a', 'b']")["products"].sum()

fig = charts.stacked_shares(
    cross,
    "nova_group",
    "nutriscore_grade",
    "share",
    NUTRISCORE_COLORS,
    segment_order=list("abcde"),
    row_order=[1, 2, 3, 4],
    row_labels=NOVA_LABELS,
    title=f"{fmt_pct(nova4_ab)} des produits ultra-transformés sont pourtant classés A ou B",
    subtitle=describe(assoc),
)
fig.update_xaxes(title_text="Part des produits du groupe NOVA (%)")
save_figure(fig, "nutriscore_by_nova")
findings.append(
    f"{fmt_int(n_nova4_ab)} produits ultra-transformés ({fmt_pct(nova4_ab)} du groupe NOVA 4) "
    f"sont classés A ou B : premiers candidats pour l'analyse des controverses."
)
findings.append(
    f"Lien Nutri-Score / NOVA : V de Cramér = {fmt_num(assoc.effect_size, 2)} "
    f"(association {assoc.interpretation})."
)
fig.show()

In [5]:
caption = "Effectifs : groupe NOVA (lignes) x Nutri-Score (colonnes)"
table.style.format("{:,.0f}").set_caption(caption)

nutriscore_grade,a,b,c,d,e
nova_group,,,,,
1,"20,710","6,174","5,802","2,500",982
2,75,"5,088","1,717",682,"5,152"
3,"9,303","6,682","14,264","18,617","14,646"
4,"8,485","11,665","37,396","42,467","52,607"


**Lecture** : le test du chi² confirme que les deux scores ne sont pas indépendants, ce qui était attendu. Le V de Cramér mesure la *force* du lien : avec plusieurs centaines de milliers de produits, la p-valeur est toujours minuscule et ne dit rien de l'importance du lien. Un lien qui n'est pas fort confirme que les deux indicateurs apportent des informations complémentaires.

## 4. Les additifs

Population : produits avec liste d'ingrédients. Un produit est compté une seule fois par additif, même si l'additif apparaît plusieurs fois.

In [6]:
top = nutrition.top_additives(con, n=15)
n_ing = round(top["products"].iloc[0] * 100 / top["share"].iloc[0])

fig = charts.ranked_bars(
    top,
    "additive",
    "share",
    suffix=" %",
    title=f"L'additif le plus courant, {top['additive'].iloc[0]}, est présent dans "
    f"{fmt_pct(top['share'].iloc[0])} des produits",
    subtitle=f"Les 15 additifs les plus fréquents, {fmt_int(n_ing)} produits "
    "avec liste d'ingrédients",
    value_label="Part des produits contenant l'additif (%)",
)
save_figure(fig, "top_additives")
fig.show()

Les noms et les évaluations de sécurité des additifs (EFSA) seront ajoutés à l'étape 5, consacrée aux controverses.

In [7]:
by_nova = nutrition.additives_by_nova(con)
pairs = nutrition.nova_additives_pairs(con)
rho = spearman(pairs["nova_group"], pairs["n_additives"])
mean4 = by_nova.set_index("nova_group").loc[4, "mean_additives"]
mean1 = by_nova.set_index("nova_group").loc[1, "mean_additives"]

fig = charts.category_bars(
    by_nova.assign(share=by_nova["mean_additives"]),
    "nova_group",
    "share",
    NOVA_COLORS,
    labels=NOVA_LABELS,
    counts="products",
    title=f"Un produit ultra-transformé contient en moyenne {fmt_num(mean4)} additifs, "
    f"contre {fmt_num(mean1)} pour un produit brut",
    subtitle=describe(rho),
    value_label="Nombre moyen d'additifs",
)
fig.update_traces(
    text=[fmt_num(v) for v in by_nova["mean_additives"]],
    hovertemplate="%{x}<br>%{y:.2f} additifs en moyenne<br>"
    "%{customdata:,.0f} produits<extra></extra>",
)
save_figure(fig, "additives_by_nova")
findings.append(
    f"Additifs : {fmt_num(mean4)} en moyenne en NOVA 4 contre {fmt_num(mean1)} en NOVA 1 "
    f"(Spearman rho = {fmt_num(rho.effect_size, 2)})."
)
fig.show()

In [8]:
by_nova.rename(
    columns={
        "nova_group": "NOVA",
        "products": "Produits",
        "mean_additives": "Moyenne",
        "median_additives": "Médiane",
        "share_without_additives": "Sans additif (%)",
    }
).style.format(
    {"Produits": "{:,.0f}", "Moyenne": "{:.2f}", "Médiane": "{:.0f}", "Sans additif (%)": "{:.1f}"}
).hide(axis="index")

NOVA,Produits,Moyenne,Médiane,Sans additif (%)
1,"44,007",0.06,0,94.6
2,"7,285",0.05,0,95.8
3,"73,100",0.31,0,78.0
4,"195,132",2.73,2,15.4


**Lecture** : le lien entre NOVA et additifs est en partie mécanique, car la présence de certains additifs suffit à classer un produit en NOVA 4. Cette corrélation confirme la cohérence des données plus qu'elle ne révèle un phénomène nouveau. C'est aussi une mise en garde pour l'étape 6 : un modèle qui prédit NOVA s'appuiera fortement sur les additifs présents dans la liste d'ingrédients.

## 5. Le Nutri-Score par rayon

Population : produits avec Nutri-Score et groupe alimentaire. Les boissons alcoolisées et les aliments pour bébés n'apparaissent pas : le Nutri-Score ne s'applique pas à ces produits.

In [9]:
by_group = nutrition.label_food_groups(nutrition.nutriscore_by_group(con))
order = by_group.drop_duplicates("food_group")["food_group_label"].tolist()
best = by_group.iloc[0]
worst = by_group.drop_duplicates("food_group").iloc[-1]

fig = charts.stacked_shares(
    by_group,
    "food_group_label",
    "nutriscore_grade",
    "share",
    NUTRISCORE_COLORS,
    segment_order=list("abcde"),
    row_order=order,
    title=f"{best['food_group_label']} : {fmt_pct(best['share_ab'])} de A ou B ; "
    f"{worst['food_group_label']} : {fmt_pct(worst['share_ab'])}",
    subtitle=f"Répartition du Nutri-Score par groupe alimentaire, "
    f"{fmt_int(by_group['products'].sum())} produits",
)
fig.update_xaxes(title_text="Part des produits du groupe (%)")
save_figure(fig, "nutriscore_by_food_group")
findings.append(
    f"Rayons : de {fmt_pct(best['share_ab'])} de A ou B ({best['food_group_label']}) "
    f"à {fmt_pct(worst['share_ab'])} ({worst['food_group_label']})."
)
fig.show()

## 6. Sucre, sel et graisses saturées par rayon

Population : produits avec groupe alimentaire, nutriment renseigné. Valeurs **médianes** pour 100 g : la moitié des produits du rayon sont au-dessus, la moitié en dessous. Un graphique par nutriment, chacun sur sa propre échelle.

In [10]:
nutrients = nutrition.label_food_groups(nutrition.nutrients_by_group(con))
units = {"sugars": "Sucres", "salt": "Sel", "saturated_fat": "Graisses saturées"}

for column, name in units.items():
    top_row = nutrients.sort_values(column, ascending=False).iloc[0]
    fig = charts.ranked_bars(
        nutrients,
        "food_group_label",
        column,
        suffix=" g",
        value_format=".1f",
        title=f"{name} : {top_row['food_group_label']} en tête, "
        f"{fmt_num(top_row[column])} g pour 100 g en médiane",
        subtitle=f"Teneur médiane pour 100 g par groupe alimentaire, "
        f"{fmt_int(nutrients['n_' + column].sum())} produits renseignés",
        value_label=f"{name} (g pour 100 g, médiane)",
    )
    save_figure(fig, f"{column}_by_food_group")
    findings.append(
        f"{name} : teneur médiane la plus élevée pour {top_row['food_group_label']} "
        f"({fmt_num(top_row[column])} g pour 100 g)."
    )
    fig.show()

## 7. Synthèse

Constats calculés à partir des données ci-dessus.

In [11]:
for line in findings:
    print("-", line)

- Nutri-Score : 55 % de D ou E, 24 % de A ou B.
- NOVA : 57 % de produits ultra-transformés (groupe 4).
- 20 150 produits ultra-transformés (13 % du groupe NOVA 4) sont classés A ou B : premiers candidats pour l'analyse des controverses.
- Lien Nutri-Score / NOVA : V de Cramér = 0,33 (association forte).
- Additifs : 2,7 en moyenne en NOVA 4 contre 0,1 en NOVA 1 (Spearman rho = 0,67).
- Rayons : de 60 % de A ou B (Fruits et légumes) à 2 % (Produits sucrés).
- Sucres : teneur médiane la plus élevée pour Produits sucrés (36,4 g pour 100 g).
- Sel : teneur médiane la plus élevée pour Viandes, poissons, œufs (1,5 g pour 100 g).
- Graisses saturées : teneur médiane la plus élevée pour Produits laitiers (10,7 g pour 100 g).


## 8. Limites

- **Sous-populations différentes** : le Nutri-Score couvre environ 40 % des produits, NOVA environ 27 %. Les pourcentages de chaque section ne portent pas sur les mêmes produits.
- **Biais de couverture** : les produits bien renseignés sur Open Food Facts sont probablement plus populaires ou issus de grandes marques. Les résultats décrivent l'offre documentée, pas les ventes réelles : un produit vendu à des millions d'exemplaires pèse autant qu'un produit confidentiel.
- **Liens mécaniques** : une partie du lien NOVA / additifs découle de la définition même de NOVA. De même, le Nutri-Score est calculé à partir du sucre, du sel et des graisses saturées.
- **Médianes par rayon** : un groupe alimentaire reste hétérogène (les produits laitiers mêlent lait, yaourts et fromages). Les comparaisons fines se feront à un niveau plus précis lorsque c'est nécessaire, notamment pour le bio à l'étape 4.

Les figures interactives sont enregistrées dans `reports/figures/`.